# A2 — Pierce corpus exploration
This notebook measures the declared PDF directly across every extracted page. It writes no derived data or generated files.
The low-DPI scan retains per-page word count, ink depth, orphan ink, centre-line span, zero-embedded-word/landscape flags, and percentile summaries.

In [1]:
import json
from pathlib import Path

import cv2
import fitz
import numpy as np

PDF_NAME = 'pierce-peoples-common-sense-medical-adviser-1890.pdf'
ROOT = next((path.resolve() for path in (Path.cwd(), Path.cwd().parent) if (path / 'configs').is_dir()), Path.cwd().resolve())
PDF_CANDIDATES = (
    ROOT / 'data' / 'raw' / PDF_NAME,
    ROOT.parent / 'data' / PDF_NAME,
)
PDF = next((path.resolve() for path in PDF_CANDIDATES if path.is_file()), PDF_CANDIDATES[0].resolve())
if not PDF.is_file():
    raise FileNotFoundError(f'{PDF} not found; run scripts/get_data.sh first')

DPI = 150

def render(page, dpi=DPI):
    pix = page.get_pixmap(dpi=dpi, alpha=False)
    array = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, pix.n)
    return cv2.cvtColor(array, cv2.COLOR_RGB2GRAY)

CORPUS_LABEL = 'pierce-1890'
doc = fitz.open(PDF)
print({'corpus': CORPUS_LABEL, 'pages': doc.page_count, 'dpi': DPI})


{'corpus': 'pierce-1890', 'pages': 1034, 'dpi': 150}


In [2]:
def page_metrics(page, index):
    words = page.get_text('words')
    gray = render(page)
    height, width = gray.shape
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    ink = binary > 0
    paper = gray[~ink]
    background = float(np.median(paper)) if paper.size else 220.0
    ink_depth = background - float(np.percentile(gray[ink], 10)) if ink.any() else 0.0

    covered = np.zeros((height, width), dtype=bool)
    sx, sy = width / page.rect.width, height / page.rect.height
    for x0, y0, x1, y1, *_ in words:
        left = max(0, min(width, int(x0 * sx)))
        top = max(0, min(height, int(y0 * sy)))
        right = max(0, min(width, int(x1 * sx) + 1))
        bottom = max(0, min(height, int(y1 * sy) + 1))
        if right > left and bottom > top:
            covered[top:bottom, left:right] = True
    orphan_ink = float((ink & ~covered).sum()) / max(int(ink.sum()), 1)

    centre = page.rect.width / 2
    centre_span = sum(x0 < centre < x1 for x0, _, x1, *_ in words) / max(len(words), 1)
    return {
        'page_id': f'p{index + 1:04d}',
        'words': len(words),
        'ink_depth': round(ink_depth, 3),
        'orphan_ink': round(orphan_ink, 6),
        'centre_span': round(centre_span, 6),
        'zero_embedded_words': len(words) == 0,
        'landscape': page.rect.width > page.rect.height,
    }

metrics = [page_metrics(page, index) for index, page in enumerate(doc)]

def percentiles(field):
    values = np.asarray([row[field] for row in metrics], dtype=float)
    return {str(q): round(float(np.percentile(values, q)), 6) for q in (0, 5, 25, 50, 75, 95, 100)}

summary = {
    'corpus': CORPUS_LABEL,
    'source_pdf': f'data/raw/{PDF_NAME}',
    'dpi': DPI,
    'pages': len(metrics),
    'embedded_words': sum(row['words'] for row in metrics),
    'min_words': min(row['words'] for row in metrics),
    'max_words': max(row['words'] for row in metrics),
    'zero_embedded_word_pages': [row['page_id'] for row in metrics if row['zero_embedded_words']],
    'landscape_pages': [row['page_id'] for row in metrics if row['landscape']],
    'percentiles': {field: percentiles(field) for field in ('words', 'ink_depth', 'orphan_ink', 'centre_span')},
}
print(json.dumps(summary, indent=2))
doc.close()


{
  "corpus": "pierce-1890",
  "source_pdf": "data/raw/pierce-peoples-common-sense-medical-adviser-1890.pdf",
  "dpi": 150,
  "pages": 1034,
  "embedded_words": 360083,
  "min_words": 0,
  "max_words": 786,
  "zero_embedded_word_pages": [
    "p0001",
    "p0002",
    "p0003",
    "p0004",
    "p0005",
    "p0006",
    "p0007",
    "p0012",
    "p0014",
    "p0691",
    "p0697",
    "p0705",
    "p0772",
    "p0898",
    "p0940",
    "p1012",
    "p1029",
    "p1030",
    "p1031",
    "p1032",
    "p1033"
  ],
  "landscape_pages": [
    "p0987",
    "p0988"
  ],
  "percentiles": {
    "words": {
      "0": 0.0,
      "5": 77.65,
      "25": 306.0,
      "50": 372.0,
      "75": 397.0,
      "95": 559.75,
      "100": 786.0
    },
    "ink_depth": {
      "0": 10.0,
      "5": 113.0,
      "25": 123.0,
      "50": 130.0,
      "75": 137.0,
      "95": 154.0,
      "100": 185.0
    },
    "orphan_ink": {
      "0": 0.082353,
      "5": 0.103879,
      "25": 0.123753,
      "50": 0.142937

The printed summary is computed from all pages in the selected PDF; `metrics` retains one measured row per page in memory for follow-up analysis. Interpret these measurements as corpus diagnostics, not OCR or layout accuracy.